# LibAR 통합 데모 (Colab GPU)

**어제 학습한 best.pt + 오늘 찍은 서가 이미지 + 이미지로 만든 장서데이터 + 한국어 OCR** 을 GPU에서 결합.

책등(이미지) + 청구기호 라벨 + 제목을 **이중인식**하여 → 오배열 판정 · 도서 검색.

## 사용법
1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. 셀을 위에서부터 실행
3. 업로드 요청 시 3개 파일 올리기: `best.pt`, `shelf_4558.jpg`, `books_4558.csv`


In [ ]:
# 1. 설치
# ⚠ paddlepaddle은 반드시 'CPU' 버전으로! (paddlepaddle-gpu는 Colab torch의 CUDA 라이브러리를
#    덮어써서 깨뜨림). CPU paddle은 nvidia 라이브러리를 안 건드려 torch(GPU)와 공존 OK.
#    → YOLO(torch)=GPU, OCR(paddle)=CPU + 인식최적화(2.7배)로 충분히 빠름
!pip install -q ultralytics paddleocr paddlepaddle
import torch; print("torch CUDA (YOLO용):", torch.cuda.is_available())
# 설치 후 반드시: 런타임 → 세션 다시 시작 → 이 셀부터 한 번씩 (또는 '모두 실행')

In [ ]:
# 2. 파일 업로드: best.pt, shelf_4558.jpg, books_4558.csv
from google.colab import files
up = files.upload()
print("업로드됨:", list(up.keys()))
IMG="shelf_4558.jpg"; CATALOG="books_4558.csv"; MODEL="best.pt"

In [ ]:
# 3. 공통 함수 (매칭·판정·정규화)
import re, csv, difflib, unicodedata
import numpy as np, cv2
from PIL import Image, ImageDraw, ImageFont

def norm(s):
    return re.sub(r"[\s\-–—_/·.]+","",unicodedata.normalize("NFC",str(s))).lower()
def norm_title(s):
    return re.sub(r"[^0-9A-Za-z가-힣]","",unicodedata.normalize("NFC",str(s))).lower()

def load_catalog(path):
    rows=list(csv.DictReader(open(path,encoding="utf-8-sig")))
    for i,r in enumerate(rows):
        r["_norm"]=norm(r["call_number"]); r["_tnorm"]=norm_title(r["title"]); r["_order"]=i
    return rows

def match_by_callnumber(text, catalog):
    # "408 뉴88 55" 에서 권차 추출 → 408-뉴88-{권차} 정확 매칭 (장서에 없으면 None)
    nums=[int(n) for n in re.findall(r"\d{1,3}", text) if n not in ("408","88","40","80","884","10","100","108")]
    for v in nums:
        cn=norm(f"408뉴88{v}")
        for r in catalog:
            if r["_norm"]==cn: return r
    return None

def match_by_title(text, catalog, thr=0.5):
    t=norm_title(text)
    if len(t)<2: return None,0.0
    best,bs=None,0.0
    for r in catalog:
        c=r["_tnorm"]
        if not c: continue
        lm=difflib.SequenceMatcher(None,t,c).find_longest_match(0,len(t),0,len(c))
        p=lm.size/min(len(t),len(c)) if min(len(t),len(c)) else 0
        s=0.6*p+0.4*difflib.SequenceMatcher(None,t,c).ratio()
        if s>bs: best,bs=r,s
    return (best,round(bs,2)) if bs>=thr else (None,round(bs,2))

def lis_misplaced(keys):
    n=len(keys)
    if n==0: return set()
    L=[1]*n; P=[-1]*n
    for i in range(n):
        for j in range(i):
            if keys[j]<=keys[i] and L[j]+1>L[i]: L[i]=L[j]+1; P[i]=j
    e=max(range(n),key=lambda i:L[i]); keep=set()
    while e!=-1: keep.add(e); e=P[e]
    return set(range(n))-keep
print("함수 준비 완료")

In [ ]:
# 4. 모델 로드 (YOLO 책등=GPU + 한국어 OCR=CPU, 인식 위주 최적화 설정)
import os
os.environ["FLAGS_use_mkldnn"] = "0"   # paddle CPU oneDNN 버그 회피
from ultralytics import YOLO
from paddleocr import PaddleOCR
yolo = YOLO(MODEL)   # torch → GPU 자동 사용
# 방향분류·문서보정 끄기 = 우리엔 불필요 + 2.7배 빠름 (로컬 실측)
ocr = PaddleOCR(lang="korean", use_doc_orientation_classify=False,
                use_doc_unwarping=False, use_textline_orientation=False,
                enable_mkldnn=False)

def ocr_read(img_bgr):
    res=ocr.predict(img_bgr)
    if not res: return "",0.0
    r0=res[0]; txts=r0.get("rec_texts",[]); scs=r0.get("rec_scores",[])
    if not txts: return "",0.0
    return " ".join(str(t) for t in txts), float(np.mean(scs))

def ocr_title(pil_crop):
    # 세로 제목 대응: 0/-90/90도 회전 시도 후 최고(글자수×신뢰도) 선택
    best=("",0.0,0.0)
    for rot in (0,-90,90):
        im=pil_crop.rotate(rot,expand=True) if rot else pil_crop
        im=im.resize((im.width*2,im.height*2), Image.LANCZOS)
        t,c=ocr_read(cv2.cvtColor(np.array(im),cv2.COLOR_RGB2BGR))
        sc=len(norm_title(t))*c
        if sc>best[2]: best=(t,c,sc)
    return best[0],best[1]
print("모델 로드 완료")

In [ ]:
# 5. 이중인식 파이프라인: 책등마다 (청구기호 라벨 + 제목) 동시 판독 → 장서 대조
import time
img=Image.open(IMG).convert("RGB")
bgr=cv2.cvtColor(np.array(img),cv2.COLOR_RGB2BGR); H,W=bgr.shape[:2]
catalog=load_catalog(CATALOG)

# YOLO 책등 탐지 → 메인 서가(라벨 띠에 걸치는 책등)만
det=yolo.predict(bgr,conf=0.2,verbose=False)[0]
spines=[[int(v) for v in b.xyxy[0].tolist()] for b in det.boxes]
band_mid=H*0.74
main=[b for b in spines if b[1]<=band_mid<=b[3]]
main.sort(key=lambda b:(b[0]+b[2])/2)
print(f"책등 {len(spines)}개 (메인 {len(main)}개), 장서 {len(catalog)}권")

t0=time.time(); books=[]
for b in main:
    x0,y0,x1,y1=b; h=y1-y0
    label = bgr[max(0,y1-int(h*0.20)):y1, x0:x1]                 # 하단=청구기호
    title = img.crop((x0, y0+int(h*0.08), x1, y0+int(h*0.55)))    # 상단=제목
    cn_text,cn_conf = ocr_read(label)
    ti_text,ti_conf = ocr_title(title)
    cn_row = match_by_callnumber(cn_text, catalog)
    ti_row,ti_s = match_by_title(ti_text, catalog)
    # ── 이중인식 결합 ──
    if cn_row and ti_row and cn_row is ti_row:
        row,ev = cn_row,"라벨+제목"            # 둘 다 일치 → 확정
    elif cn_row and ti_row and cn_row is not ti_row:
        row,ev = ti_row,"제목(라벨 불일치)"     # 충돌 → 제목 우선(더 변별력↑)
    elif cn_row:
        row,ev = cn_row,"라벨"
    elif ti_row:
        row,ev = ti_row,"제목(복구)"           # 청구기호 오독을 제목으로 복구
    else:
        row,ev = None,"미인식"
    books.append({"box":b,"cn_text":cn_text,"cn_conf":round(cn_conf,2),
                  "ti_text":ti_text,"call_number":row["call_number"] if row else None,
                  "title":row["title"] if row else None,
                  "order":row["_order"] if row else None,"evidence":ev})
print(f"판독 {len(books)}권, {time.time()-t0:.1f}s")

# 오배열 판정 (장서 정렬 순서 위배 = LIS 밖)
matched=[b for b in books if b["call_number"]]
mis=lis_misplaced([b["order"] for b in matched])
for b in books: b["status"]="unknown" if b["call_number"] is None else "ok"
for i,b in enumerate(matched):
    if i in mis: b["status"]="misplaced"
dual=sum(1 for b in books if b["evidence"]=="라벨+제목")
rec =sum(1 for b in books if b["evidence"]=="제목(복구)")
print(f"이중확인 {dual}권 · 제목복구 {rec}권 · 오배열 {len(mis)}권 · 미인식 {sum(1 for b in books if b['status']=='unknown')}")

In [ ]:
# 6. 렌더 함수 + 사서 모드(서가 점검) 결과 이미지
COLORS={"ok":(40,190,90),"misplaced":(235,60,60),"unknown":(150,150,150),"found":(0,200,220)}
KO={"ok":"정상","misplaced":"오배열!","unknown":"미인식","found":"찾는 책"}
def get_font(sz):
    for p in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]:
        try: return ImageFont.truetype(p,sz)
        except: pass
    return ImageFont.load_default()
# 한글 폰트 설치 (Colab)
!apt-get -qq install fonts-nanum >/dev/null 2>&1

def render(books, banner, out):
    vis=img.copy(); d=ImageDraw.Draw(vis,"RGBA")
    f=get_font(46); fs=get_font(30)
    for b in books:
        x0,y0,x1,y1=b["box"]; st=b["status"]; c=COLORS[st]
        w=8 if st in ("misplaced","found") else 5
        d.rectangle([x0,y0,x1,y1],outline=c+(255,),width=w)
        if st in ("misplaced","found"): d.rectangle([x0,y0,x1,y1],fill=c+(70,))
        tag=(b["call_number"] or "?").split("-")[-1]
        d.rectangle([x0,y1-52,x0+70,y1],fill=c+(235,)); d.text((x0+5,y1-50),tag,font=fs,fill=(255,255,255,255))
    d.rectangle([0,0,W,150],fill=(20,30,60,225)); d.text((40,30),banner,font=f,fill=(255,255,255,255))
    vis.save(out,quality=90); return out

n_ok=sum(1 for b in books if b["status"]=="ok"); n_mis=sum(1 for b in books if b["status"]=="misplaced")
render(books, f"LibAR 서가 점검 | 이중인식 {len(matched)}권 (정상 {n_ok} · 오배열 {n_mis})", "out_inspect.jpg")
from IPython.display import Image as IPImage, display
display(IPImage("out_inspect.jpg", width=1100))

In [ ]:
# 7. 이용자 모드(도서 검색) — 제목/청구기호로 찾아 하이라이트
QUERY = "자연의 기하학"   # ← 찾을 책 (제목 또는 청구기호)
q=norm_title(QUERY); qc=norm(QUERY); found=None
for b in books:
    b2=dict(b)
    if b["call_number"] and (q and q in norm_title(b["title"] or "") or qc and qc in norm(b["call_number"])):
        b["status"]="found"; found=b; break
banner=(f"LibAR 도서 검색 | '{QUERY}' → 찾음: {found['call_number']} ({found['title']})"
        if found else f"LibAR 도서 검색 | '{QUERY}' → 서가에 없음")
render(books, banner, "out_search.jpg")
display(IPImage("out_search.jpg", width=1100))
for b in books:
    if b["status"]=="found": b["status"]="ok"   # 원복

In [ ]:
# 8. 결과 다운로드
from google.colab import files
files.download("out_inspect.jpg")
files.download("out_search.jpg")

---
## 대림도서관 적용
`books_4558.csv` 를 **솔로몬 장서데이터**(청구기호·제목)로 교체하면 그대로 정확해집니다.
청구기호↔제목의 정확한 연결이 솔로몬에 있으므로, 이중인식 정확도가 완성됩니다.
